This script is to generate lca file for FA checks. 
Inputs: 
1. an existing lca file with two basic CM1 LC - N+ and N-
2. list of joints to which FA will apply

Output: updated lca file with generated LC for each joint



In [27]:
# load existing lca
from pathlib import Path 
lca_input = Path("tower_36_input.lca")
lca_output = Path("tower_36_FA.lca")
joints_list = Path("tower_36_jointlist.txt")

Empty_line = "0.0000000000 0.0000000000 0.0000000000 ; V T and L loads including insul weight"
FA_line = "21000.0000000000 0.0000000000 0.0000000000 '' ; V T and L loads including insul weight"

In [28]:
# read lca file
with open(lca_input, "r") as f:
    lca_data = f.read()

# separate header data - first 4 lines
lca_header = lca_data.splitlines()[:4]

# start reading from 5th line
lca_lines = lca_data.splitlines()[4:]
# search for  line number that has only "0" 
for i, line in enumerate(lca_lines):
    if line.strip() == "0":
        break  

print(lca_lines[0], i)


CM1+E NA+,T NA+ 62


In [29]:
# work with header data
for line in lca_header:
    print(line) 

# second line is the number of LC,
# third line is the number of applyed loads
# save them both as variables
num_lc = int(lca_header[1].split(";", 1)[0].strip())
num_applied_loads = int(lca_header[2].split(";", 1)[0].strip())

print(f"Number of Load Cases: {num_lc}")
print(f"Number of Applied Loads: {num_applied_loads}")

# later we will need to update both numbers and write them back to the header


TYPE='LCA FILE' VERSION='17' UNITS='SI' SOURCE='Tower Version 21.01' USER='Alliance Power & Data - Australia' FILENAME=''
2 ; num_load_cases
28 ; num_load_points
0 1 1 1 0.142857 10 1 0.740000 1 0 1 1 0 0 1 0 0 1.600000 0 0 1 39.969000 10000.000000 3
Number of Load Cases: 2
Number of Applied Loads: 28


In [30]:
# create list with first basic LC:
lc_basic_1 = lca_lines[0:i]

In [31]:
# repeat same for second basic LC:
for j, line in enumerate(lca_lines[i+1:], start=i+1):
    if line.strip() == "0":
        break
lc_basic_2 = lca_lines[i+1:j]

print(lca_lines[i+1], j)

CM1+E NA-,T NA- 125


now we have two basic LC, which we will copy for each joint

save joint list in a txt file and save in same folder


In [32]:
# read joint list
with open(joints_list, "r") as f:
    joint_data = f.read()
    joint_list = [line.strip() for line in joint_data.split("\n") if line.strip()]  

print(joint_list)
#print number of joints
num_of_joints = len(joint_list)*2
print(f"Number of joints: {num_of_joints}")


['820X', '720X', '620X', '820P', '720P', '620P', '821X', '721X', '621X', '821S', '721S', '621S', '821XY', '721XY', '621XY', '821Y', '721Y', '621Y', '822X', '722X', '622X', '822S', '722S', '622S', '822XY', '722XY', '622XY', '822Y', '722Y', '622Y', '823X', '723X', '623X', '823S', '723S', '623S', '823XY', '723XY', '623XY', '823Y', '723Y', '623Y']
Number of joints: 84


now for each joint create new LC_1 and LC_2 with joint name and body of lc_basic_1.

new LC_1 will have lines:

    joint name, "+" 
    lc_basic_1 lines
    joint name
    FA_line
    "0"


then same with lc_basic_2:

    joint name, "-" 
    lc_basic_1 lines
    joint name
    FA_line
    "0"
    


In [33]:
# create output file 
# wrire midified header with updated number of LC and applied loads
with open(lca_output, "w") as f:
    # update header with new number of LC and applied loads
    lca_header[1] = f"{num_lc + num_of_joints} ; Number of Load Cases"
    lca_header[2] = f"{num_applied_loads + 1} ; Number of Applied Loads"
    
    # write updated header to output file
    f.write("\n".join(lca_header) + "\n")
    
    # write basic LC with empty line
    f.write("\n".join(lc_basic_1) + "\n")
    f.write("\n")
    f.write(Empty_line + "\n")
    f.write("0" + "\n")
    f.write("\n".join(lc_basic_2) + "\n")
    f.write("\n")
    f.write(Empty_line + "\n")
    f.write("0" + "\n")

The final part

In [34]:
# now for each joint in list
for joint in joint_list:
    lc1 = lc_basic_1.copy()
    lc1[0] = joint + " + "
    lc1.append(joint)
    lc1.append(FA_line)
    lc1.append("0")

    lc2 = lc_basic_2.copy()
    lc2[0] = joint + " - "
    lc2.append(joint)
    lc2.append(FA_line)
    lc2.append("0")

    # write new lca file
    with open(lca_output, "a") as f:
        f.write("\n".join(lc1) + "\n")
        f.write("\n".join(lc2) + "\n")